# Legacy repeated-run classification on shared cohorts

This notebook modernizes the FlowCode repeated split experiment. A single checksummed registry is used by every LOT classifier, aggregation scheme, FlowSOM, and deep set/point-cloud model. Within each repeat, the test cohort is fixed and the balanced training cohorts are nested: **2 ⊂ 4 ⊂ 6 ⊂ 8 patients per class**.

In [ ]:
from pathlib import Path
import json
import h5py
import pandas as pd
from IPython.display import Image, display

from flowlot.evaluation.repeated_benchmark import (
    aggregate_results, audit_registry, create_job_table, create_split_registry,
    export_legacy_splits_h5, run_job,
)

## 1. Experiment coordinates

Use the patient intersection when every classifier must see the same cohort across all tubes. `PREPROCESS` is the ID after `preprocess_`; `EMBEDDING` is the LOT group name such as `patient0_sinkhorn`.

In [ ]:
STAGE2 = Path('../stage2_analytics.h5')
DATASET, CELL_COUNT = 'BLAST110', '1000'
TUBES = ['P1', 'P2', 'P3', 'P4']
PREPROCESS, EMBEDDING = 'common12', 'patient0_sinkhorn'
RESULTS = Path('../results/repeated_classification')
TRAIN_PER_CLASS = (2, 4, 6, 8)
REPEATS, TEST_SIZE, SEED = 10, 0.5, 42
RESULTS.mkdir(parents=True, exist_ok=True)

## 2. Create and audit the shared split files

The JSON file is authoritative. The HDF5 export retains the exact historical `run_N/subsamples/num_sub_per_cls_K` paths so older FlowCode readers can consume the corrected nested cohorts.

In [ ]:
registry = create_split_registry(
    STAGE2, DATASET, CELL_COUNT, RESULTS / 'shared_splits.json',
    train_per_class=TRAIN_PER_CLASS, repeats=REPEATS, test_size=TEST_SIZE,
    seed=SEED, tubes=TUBES, patient_policy='intersection',
)
export_legacy_splits_h5(registry, RESULTS / 'legacy_splits.h5')
audit = pd.DataFrame(audit_registry(registry))
display(audit.head(8))
print('Registry SHA-256:', registry['registry_hash'])

In [ ]:
for split in registry['splits']:
    cohorts = [set(split['train_ids_by_k'][str(k)]) for k in TRAIN_PER_CLASS]
    assert all(left < right for left, right in zip(cohorts, cohorts[1:]))
    assert all(not (cohort & set(split['test_ids'])) for cohort in cohorts)
with h5py.File(RESULTS / 'legacy_splits.h5') as h5:
    print('Legacy paths:', list(h5['run_0/subsamples']))
    assert h5.attrs['flowlot_registry_hash'] == registry['registry_hash']
print('All repeats are balanced, nested, disjoint, and checksum-linked.')

## 3. Generate the classifier × aggregation × repeat × k job matrix

Deep cell models run once per tube. LOT-vector classifiers run per tube and with early/late multi-tube aggregation. Every row points back to the same registry.

In [ ]:
models = [
    'logistic', 'linear_svm', 'random_forest', 'extra_trees', 'nsc', 'nsc_energy',
    'xgboost',  # Optional: pip install -e '.[xgboost]'
    'flowsom', 'cellcnn', 'attention_mil', 'cytoset', 'dgcnn', 'pointnet2',
]
jobs = create_job_table(
    RESULTS / 'shared_splits.json', RESULTS / 'jobs.tsv', models,
    aggregations=('single', 'early_mean', 'early_zero', 'late_soft'),
)
jobs_frame = pd.DataFrame(jobs)
display(jobs_frame.groupby(['model', 'aggregation', 'tube']).size().rename('jobs').reset_index())
print(f'{len(jobs):,} independently resumable jobs')

## 4. Run on Slurm (recommended)

Copy and edit `scripts/repeated_benchmark.env.example`, then run:

```bash
export FLOWLOT_CONFIG=/absolute/path/repeated_benchmark.env
bash scripts/hpc_repeated_benchmark.sh init
bash scripts/hpc_repeated_benchmark.sh submit
bash scripts/hpc_repeated_benchmark.sh aggregate
```

Each array task writes one atomic JSON shard. Re-running the array skips completed shards. For a quick local smoke test, call `run_job(..., index=0)` below.

In [ ]:
RUN_ONE_LOCAL_JOB = False
if RUN_ONE_LOCAL_JOB:
    shard = run_job(
        STAGE2, RESULTS / 'shared_splits.json', RESULTS / 'jobs.tsv', RESULTS,
        PREPROCESS, EMBEDDING, index=0, epochs=2, max_cells=512,
    )
    print(shard)

## 5. Validate and compare accumulated results

Aggregation fails closed when a shard is missing or comes from another registry. Set `ALLOW_INCOMPLETE=True` only while monitoring an active array. Bootstrap intervals treat each patient as one unit: probabilities from repeated test appearances are averaged, followed by stratified within-class resampling.

In [ ]:
ALLOW_INCOMPLETE = False
integrity = aggregate_results(
    RESULTS / 'shared_splits.json', RESULTS / 'jobs.tsv', RESULTS / 'shards',
    RESULTS / 'aggregate', allow_incomplete=ALLOW_INCOMPLETE,
    bootstrap_iterations=2000, confidence_level=0.95, bootstrap_seed=42,
)
print(json.dumps(integrity, indent=2))
summary = pd.read_csv(RESULTS / 'aggregate/summary.csv')
bootstrap = pd.read_csv(RESULTS / 'aggregate/bootstrap_ci.csv')
paired = pd.read_csv(RESULTS / 'aggregate/paired_comparisons.csv')
display(summary.sort_values(['k', 'balanced_accuracy_mean'], ascending=[True, False]).head(20))
display(bootstrap.sort_values(['k', 'balanced_accuracy_estimate'], ascending=[True, False]).head(20))
display(paired.sort_values('mean_delta_a_minus_b', ascending=False).head(20))
display(Image(filename=RESULTS / 'aggregate/aggregation_comparison.png'))